# TrackML Hit-Graph Edge Classification

**Task.** For one simulated collision event from the [TrackML particle-tracking challenge](https://www.kaggle.com/competitions/trackml-particle-identification), build a k-NN graph over detector hits (inner barrel, r < 200 mm) and classify each candidate edge as *"both hits belong to the same particle"* (1) or *not* (0).

**What this notebook is, and is not.** It is hit-pair (edge) classification, a first stage of track finding. It does **not** assemble tracks or compute the official TrackML score.

**Pipeline.** load events → build graphs → train two edge classifiers on one event → pick the decision threshold on a validation event → report metrics on unseen test events.

| Split | Events | Used for |
|---|---|---|
| Train | first `N_TRAIN` events | fitting weights and feature scalers |
| Validation | next `N_VAL` events | choosing model and decision threshold |
| Test | next `N_TEST` events | final reported metrics (never used for any choice) |

## 1. Setup and configuration

In [ ]:
try:
    import torch_geometric  # noqa: F401
except ImportError:
    %pip install -q torch-geometric

import glob, json, os, random, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

# ---- configuration -------------------------------------------------------
SEED = 42
R_MAX = 200            # keep hits with r < R_MAX mm (inner barrel)
K_NEIGHBORS = 5        # candidate edges per hit
N_TRAIN, N_VAL, N_TEST = 1, 1, 5
HIDDEN = 64
EPOCHS = 60
LR, WEIGHT_DECAY = 3e-3, 1e-4
POS_WEIGHT_CAP = 12.0  # cap on the positive-class loss weight
DATA_ROOT = os.environ.get("TRACKML_DATA_ROOT")   # optional: folder that contains event*-hits.csv
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print("device:", DEVICE)

## 2. Locate the data and split events
Events are sorted by file name, so the split is identical on every run.

In [ ]:
def find_event_files():
    roots = [DATA_ROOT] if DATA_ROOT else []
    roots.append(os.path.join(WORK_DIR, "train_sample"))
    for root in roots:
        files = sorted(glob.glob(os.path.join(root, "**", "event*-hits.csv"), recursive=True))
        if files:
            return files
    # Kaggle: extract train_sample.zip from the attached competition
    zip_path = next((os.path.join(r, "train_sample.zip")
                     for r, _, fs in os.walk("/kaggle/input") if "train_sample.zip" in fs), None)
    if zip_path is None:
        raise FileNotFoundError("No TrackML events found. Attach the competition data or set TRACKML_DATA_ROOT.")
    out = os.path.join(WORK_DIR, "train_sample")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out)
    return sorted(glob.glob(os.path.join(out, "**", "event*-hits.csv"), recursive=True))

all_event_files = find_event_files()
need = N_TRAIN + N_VAL + N_TEST
assert len(all_event_files) >= need, f"need {need} events, found {len(all_event_files)}"

train_files = all_event_files[:N_TRAIN]
val_files   = all_event_files[N_TRAIN:N_TRAIN + N_VAL]
test_files  = all_event_files[N_TRAIN + N_VAL:need]
print(f"{len(all_event_files)} events found -> train {len(train_files)}, val {len(val_files)}, test {len(test_files)}")
for name, fs in [("train", train_files), ("val", val_files), ("test", test_files)]:
    print(f"  {name}: {[os.path.basename(f).split('-')[0] for f in fs]}")

## 3. Quick look at the data (optional)
Interactive 3D view of 15 simulated particle trajectories from the first training event.

In [ ]:
import plotly.graph_objects as go

_h = pd.read_csv(train_files[0])
_t = pd.read_csv(train_files[0].replace("-hits.csv", "-truth.csv"))
_df = _h.merge(_t[["hit_id", "particle_id"]], on="hit_id")
_ids = _df.loc[_df.particle_id != 0, "particle_id"].unique()[:15]

fig = go.Figure()
for pid in _ids:
    p = _df[_df.particle_id == pid]
    fig.add_trace(go.Scatter3d(x=p.x, y=p.y, z=p.z, mode="lines+markers",
                               marker=dict(size=3), line=dict(width=2), name=f"Particle {pid}"))
fig.update_layout(title="Simulated particle trajectories (TrackML)", width=800, height=600,
                  scene=dict(xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)"))
fig.show()
fig.write_html(os.path.join(WORK_DIR, "particle_3d_tracks.html"))
del _h, _t, _df

## 4. Graph construction
Each hit is a node with 6 features `(x, y, z, r, φ, η)`. Each hit is linked to its 5 nearest neighbours in `(r, 100·φ, 0.1·z)` space. Each edge has 6 features `(dx, dy, dz, dr, dφ, dist)`. An edge is positive when both hits belong to the same non-noise particle.

Feature scalers are fitted on the **training** events only and reused for every other event.

In [ ]:
def load_event_raw(hits_path, r_max=R_MAX, k=K_NEIGHBORS):
    """Read one event and return raw (unscaled) node/edge arrays and edge labels."""
    truth_path = hits_path.replace("-hits.csv", "-truth.csv")
    hits = pd.read_csv(hits_path, usecols=["hit_id", "x", "y", "z"])
    truth = pd.read_csv(truth_path, usecols=["hit_id", "particle_id"])
    df = hits.merge(truth, on="hit_id")
    if r_max is not None:
        df = df[np.hypot(df["x"], df["y"]) < r_max]
    df = df.reset_index(drop=True)

    x, y, z = df["x"].values, df["y"].values, df["z"].values
    r = np.hypot(x, y)
    phi = np.arctan2(y, x)
    eta = np.arcsinh(z / (r + 1e-6))
    node_raw = np.column_stack([x, y, z, r, phi, eta])

    knn_space = np.column_stack([r, phi * 100, z * 0.1])
    idx = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1).fit(knn_space).kneighbors(knn_space, return_distance=False)
    src = np.repeat(np.arange(len(df)), k)
    dst = idx[:, 1:].ravel()                      # drop column 0 (the hit itself)

    dx, dy, dz = x[dst] - x[src], y[dst] - y[src], z[dst] - z[src]
    dr = r[dst] - r[src]
    dphi = np.arctan2(np.sin(phi[dst] - phi[src]), np.cos(phi[dst] - phi[src]))
    dist = np.sqrt(dx**2 + dy**2 + dz**2)
    edge_raw = np.column_stack([dx, dy, dz, dr, dphi, dist])

    pid = df["particle_id"].values
    label = ((pid[src] == pid[dst]) & (pid[src] != 0)).astype(np.float32)
    return dict(path=hits_path, node_raw=node_raw, edge_index=np.vstack([src, dst]),
                edge_raw=edge_raw, label=label)

def to_graph(raw, node_scaler, edge_scaler):
    return Data(x=torch.tensor(node_scaler.transform(raw["node_raw"]), dtype=torch.float),
                edge_index=torch.tensor(raw["edge_index"], dtype=torch.long),
                edge_attr=torch.tensor(edge_scaler.transform(raw["edge_raw"]), dtype=torch.float),
                y=torch.tensor(raw["label"], dtype=torch.float))

train_raw = [load_event_raw(f) for f in train_files]
node_scaler = StandardScaler().fit(np.vstack([g["node_raw"] for g in train_raw]))
edge_scaler = StandardScaler().fit(np.vstack([g["edge_raw"] for g in train_raw]))

def make_graphs(files, raws=None):
    raws = raws or [load_event_raw(f) for f in files]
    return [to_graph(r, node_scaler, edge_scaler).to(DEVICE) for r in raws]

train_graphs = make_graphs(train_files, train_raw)
val_graphs   = make_graphs(val_files)
test_graphs  = make_graphs(test_files)

summary = pd.DataFrame(
    [{"split": s, "event": os.path.basename(f).split("-")[0], "nodes": g.num_nodes,
      "edges": g.num_edges, "positive_edges": int(g.y.sum()), "positive_rate": float(g.y.mean())}
     for s, fs, gs in [("train", train_files, train_graphs), ("val", val_files, val_graphs),
                       ("test", test_files, test_graphs)] for f, g in zip(fs, gs)])
summary

## 5. Models
Both models score an edge from the embeddings of its two endpoint hits and its own edge features.

* **`EdgeMLPClassifier`**: encodes nodes and edges independently, then classifies `[source, target, edge]`. This is the best-performing model from the experiments. It has **no message-passing step**, so it is an edge-feature MLP and not a GNN.
* **`GCNEdgeClassifier`**: the same head, but node embeddings first go through two `GCNConv` layers, so each hit sees its neighbourhood. This is the genuine graph-neural-network variant.

In [ ]:
def _block(i, o):
    return nn.Sequential(nn.Linear(i, o), nn.LayerNorm(o), nn.ReLU())

class EdgeMLPClassifier(nn.Module):
    def __init__(self, node_in=6, edge_in=6, hidden=HIDDEN):
        super().__init__()
        self.node_enc = _block(node_in, hidden)
        self.edge_enc = _block(edge_in, hidden)
        self.head = nn.Sequential(nn.Linear(3 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(hidden, 1))

    def forward(self, x, edge_index, edge_attr):
        h = self.node_enc(x)
        src, dst = edge_index
        return self.head(torch.cat([h[src], h[dst], self.edge_enc(edge_attr)], dim=-1)).squeeze(-1)

class GCNEdgeClassifier(nn.Module):
    def __init__(self, node_in=6, edge_in=6, hidden=HIDDEN):
        super().__init__()
        self.node_enc = _block(node_in, hidden)
        self.edge_enc = _block(edge_in, hidden)
        self.conv1 = GCNConv(hidden, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.head = nn.Sequential(nn.Linear(3 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(hidden, 1))

    def forward(self, x, edge_index, edge_attr):
        h = self.node_enc(x)
        h = F.relu(self.conv1(h, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        src, dst = edge_index
        return self.head(torch.cat([h[src], h[dst], self.edge_enc(edge_attr)], dim=-1)).squeeze(-1)

MODELS = {"EdgeMLPClassifier": EdgeMLPClassifier, "GCNEdgeClassifier": GCNEdgeClassifier}

## 6. Training
Class-weighted binary cross-entropy (positives are about 1 in 7 edges), AdamW and a cosine learning-rate schedule, full-batch.

In [ ]:
def train_model(model, graphs, epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY):
    num_pos = sum(g.y.sum().item() for g in graphs)
    num_neg = sum((g.y == 0).sum().item() for g in graphs)
    pos_weight = torch.tensor([min(num_neg / max(num_pos, 1.0), POS_WEIGHT_CAP)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = []
    model.train()
    for epoch in range(1, epochs + 1):
        total = 0.0
        for g in graphs:
            optimizer.zero_grad()
            loss = criterion(model(g.x, g.edge_index, g.edge_attr), g.y)
            loss.backward()
            optimizer.step()
            total += loss.item()
        scheduler.step()
        history.append(total / len(graphs))
        if epoch == 1 or epoch % 10 == 0:
            print(f"  epoch {epoch:02d}/{epochs} | loss {history[-1]:.4f}")
    return history

trained, histories = {}, {}
for name, cls in MODELS.items():
    print(f"Training {name}")
    set_seed()
    trained[name] = cls().to(DEVICE)
    histories[name] = train_model(trained[name], train_graphs)

## 7. Evaluation
* **ROC-AUC** is threshold-free and is the main headline number.
* **Precision / recall / F1** depend on the decision threshold. The threshold is chosen by maximising F1 on the **validation** event, then applied unchanged to the test events.
* Metrics on the training event are shown only to expose the generalisation gap.

In [ ]:
@torch.no_grad()
def predict(model, g):
    model.eval()
    return torch.sigmoid(model(g.x, g.edge_index, g.edge_attr)).cpu().numpy(), g.y.cpu().numpy()

def edge_metrics(y_true, probs, thr):
    pred = (probs >= thr).astype(int)
    return {"roc_auc": roc_auc_score(y_true, probs),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0)}

def choose_threshold(model, graphs, grid=np.arange(0.05, 0.96, 0.05)):
    preds = [predict(model, g) for g in graphs]
    probs = np.concatenate([p for p, _ in preds]); y = np.concatenate([t for _, t in preds])
    f1s = [f1_score(y, (probs >= t).astype(int), zero_division=0) for t in grid]
    return float(grid[int(np.argmax(f1s))]), roc_auc_score(y, probs)

def evaluate(model, graphs, thr):
    per_event = pd.DataFrame([edge_metrics(*reversed(predict(model, g)), thr) for g in graphs])
    pooled = [predict(model, g) for g in graphs]
    cm = confusion_matrix(np.concatenate([t for _, t in pooled]),
                          (np.concatenate([p for p, _ in pooled]) >= thr).astype(int))
    return per_event, cm

results, rows = {}, []
for name, model in trained.items():
    thr, val_auc = choose_threshold(model, val_graphs)
    test_df, cm = evaluate(model, test_graphs, thr)
    train_df, _ = evaluate(model, train_graphs, thr)
    results[name] = {"threshold": thr, "val_roc_auc": val_auc,
                     "test_mean": test_df.mean().to_dict(), "test_std": test_df.std(ddof=0).to_dict(),
                     "train_in_sample": train_df.mean().to_dict(), "test_confusion_matrix": cm.tolist()}
    rows.append({"model": name, "threshold": thr, "val ROC-AUC": round(val_auc, 4),
                 **{f"test {k}": f"{test_df[k].mean():.4f} ± {test_df[k].std(ddof=0):.4f}" for k in test_df.columns},
                 "train ROC-AUC (in-sample)": round(train_df['roc_auc'].mean(), 4)})
    print(f"\n=== {name} (threshold {thr:.2f}) ===")
    print(test_df.round(4).rename_axis("test event").to_string())
    print("pooled test confusion matrix [[TN FP] [FN TP]]:\n", cm)

pd.DataFrame(rows).set_index("model").T

## 8. Select the final model and save outputs
The final model is chosen by **validation** ROC-AUC, so the test events play no part in the choice.

In [ ]:
best_name = max(results, key=lambda n: results[n]["val_roc_auc"])
best = results[best_name]
print(f"Final model: {best_name}  (threshold {best['threshold']:.2f})")
for k in ["roc_auc", "precision", "recall", "f1"]:
    print(f"  test {k:<9}: {best['test_mean'][k]:.4f} ± {best['test_std'][k]:.4f}")

torch.save({"model": best_name, "state_dict": trained[best_name].state_dict(),
            "threshold": best["threshold"], "config": {"R_MAX": R_MAX, "K": K_NEIGHBORS, "HIDDEN": HIDDEN},
            "node_scaler": (node_scaler.mean_, node_scaler.scale_),
            "edge_scaler": (edge_scaler.mean_, edge_scaler.scale_)},
           os.path.join(WORK_DIR, "edge_classifier.pt"))
with open(os.path.join(WORK_DIR, "results.json"), "w") as f:
    json.dump({"final_model": best_name, "models": results,
               "split": {"train": [os.path.basename(p) for p in train_files],
                         "val": [os.path.basename(p) for p in val_files],
                         "test": [os.path.basename(p) for p in test_files]}}, f, indent=2)
print("saved edge_classifier.pt and results.json to", os.path.abspath(WORK_DIR))

## Limitations
* Edge classification only. Tracks are not assembled and the official TrackML score is not computed.
* Training uses one event (`N_TRAIN = 1`); increase it to train on more.
* Only the inner barrel (r < 200 mm) is used; hits outside it are ignored.
* Data are simulated, not real LHC collisions.
* The test events come from the same 100-event sample, so they say nothing about real detector data.